In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128

model_name = "VGG16"
#model = cifar10()
model = VGG16()

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...


# Part 1

In [2]:
fdir = 'result/VGG16/model_best.pth.tar'

checkpoint = torch.load(fdir)
model.load_state_dict(checkpoint['state_dict'])


criterion = nn.CrossEntropyLoss().cuda()

model.eval()
model.cuda()


prec = validate(testloader, model, criterion)

Test: [0/79]	Time 5.778 (5.778)	Loss 0.2689 (0.2689)	Prec 92.188% (92.188%)
 * Prec 90.290% 


# Part 2

In [10]:
model2 = VGG16()
model2.load_state_dict(model.state_dict())
# model2.features[0].weight.abs().sum()

<All keys matched successfully>

In [13]:

model_name2 = "VGG16_loss2"

def train2(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss1 = criterion(output, target)
        loss2 = model.features[0].weight.abs().sum()
        loss = loss1 + loss2

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))


def validate2(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss1 = loss = criterion(output, target)
            loss2 = model.features[0].weight.abs().sum()

            loss = loss1 + loss2
            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


In [14]:
# This cell won't be given, but students will complete the training

lr = 4e-2
weight_decay = 1e-4
epochs = 30
best_prec = 0

#model = nn.DataParallel(model).cuda()
model2.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model2.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name2)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train2(trainloader, model2, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate2(testloader, model2, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model2.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 5.107 (5.107)	Data 4.981 (4.981)	Loss 79.9915 (79.9915)	Prec 57.031% (57.031%)
Epoch: [0][100/391]	Time 0.032 (0.086)	Data 0.000 (0.050)	Loss 81.2513 (75.5234)	Prec 57.031% (61.665%)
Epoch: [0][200/391]	Time 0.036 (0.059)	Data 0.001 (0.025)	Loss 81.8488 (77.3165)	Prec 55.469% (62.002%)
Epoch: [0][300/391]	Time 0.033 (0.050)	Data 0.000 (0.017)	Loss 79.0577 (77.8775)	Prec 59.375% (62.266%)
Validation starts
Test: [0/79]	Time 5.037 (5.037)	Loss 78.1436 (78.1436)	Prec 36.719% (36.719%)
 * Prec 34.740% 
best acc: 34.740000
Epoch: [1][0/391]	Time 5.176 (5.176)	Data 5.094 (5.094)	Loss 77.2387 (77.2387)	Prec 60.156% (60.156%)
Epoch: [1][100/391]	Time 0.033 (0.086)	Data 0.001 (0.051)	Loss 77.5464 (78.4864)	Prec 61.719% (63.776%)
Epoch: [1][200/391]	Time 0.033 (0.060)	Data 0.001 (0.026)	Loss 77.1198 (78.3186)	Prec 57.812% (64.556%)
Epoch: [1][300/391]	Time 0.033 (0.051)	Data 0.000 (0.017)	Loss 80.4496 (77.9987)	Prec 60.938% (64.968%)
Validation starts
Test: [0/79]	Time 5.0

# Part 3

In [15]:
def train3(trainloader, model, criterion, optimizer, epoch, gamma):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss1 = criterion(output, target)
        loss2 = model.features[0].weight.abs().sum()
        loss = loss1 + gamma * loss2

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

def validate3(val_loader, model, criterion, gamma):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss1 = loss = criterion(output, target)
            loss2 = model.features[0].weight.abs().sum()

            loss = loss1 + loss2 * gamma
            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


## Try $\gamma = 0.1$

In [16]:
model3 = VGG16()
model3.load_state_dict(model.state_dict())
model_name3 = "VGG16_loss3_gamma_e-1"

In [17]:
lr = 4e-2
weight_decay = 1e-4
epochs = 30
best_prec = 0
gamma = 1e-1
#model = nn.DataParallel(model).cuda()
model3.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model3.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name3)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train3(trainloader, model3, criterion, optimizer, epoch, gamma)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate3(testloader, model3, criterion, gamma)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model3.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 5.273 (5.273)	Data 5.188 (5.188)	Loss 20.2330 (20.2330)	Prec 100.000% (100.000%)
Epoch: [0][100/391]	Time 0.032 (0.088)	Data 0.001 (0.052)	Loss 5.6004 (8.7747)	Prec 13.281% (19.284%)
Epoch: [0][200/391]	Time 0.033 (0.060)	Data 0.000 (0.026)	Loss 5.0481 (7.7290)	Prec 14.844% (16.231%)
Epoch: [0][300/391]	Time 0.032 (0.051)	Data 0.001 (0.018)	Loss 4.9100 (7.1467)	Prec 19.531% (17.540%)
Validation starts
Test: [0/79]	Time 4.989 (4.989)	Loss 4.9501 (4.9501)	Prec 20.312% (20.312%)
 * Prec 20.630% 
best acc: 20.630000
Epoch: [1][0/391]	Time 5.143 (5.143)	Data 5.063 (5.063)	Loss 4.7321 (4.7321)	Prec 22.656% (22.656%)
Epoch: [1][100/391]	Time 0.033 (0.086)	Data 0.000 (0.051)	Loss 4.6984 (4.8432)	Prec 26.562% (26.880%)
Epoch: [1][200/391]	Time 0.032 (0.060)	Data 0.001 (0.026)	Loss 4.0125 (4.7141)	Prec 34.375% (28.906%)
Epoch: [1][300/391]	Time 0.034 (0.051)	Data 0.000 (0.017)	Loss 4.9655 (4.7609)	Prec 37.500% (30.113%)
Validation starts
Test: [0/79]	Time 5.075 (5.075)	Los

# Try $\gamma=10^{-6}$

In [18]:
model4 = VGG16()
model4.load_state_dict(model.state_dict())
model_name4 = "VGG16_loss3_gamma_e-6"

In [19]:
lr = 4e-2
weight_decay = 1e-4
epochs = 20
best_prec = 0
gamma = 1e-6
#model = nn.DataParallel(model).cuda()
model4.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model4.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name4)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train3(trainloader, model4, criterion, optimizer, epoch, gamma)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate3(testloader, model4, criterion, gamma)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model4.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 5.294 (5.294)	Data 5.209 (5.209)	Loss 0.0904 (0.0904)	Prec 96.875% (96.875%)
Epoch: [0][100/391]	Time 0.034 (0.086)	Data 0.001 (0.052)	Loss 0.8204 (1.1153)	Prec 68.750% (64.573%)
Epoch: [0][200/391]	Time 0.036 (0.060)	Data 0.001 (0.027)	Loss 0.6589 (0.9221)	Prec 81.250% (70.406%)
Epoch: [0][300/391]	Time 0.033 (0.051)	Data 0.001 (0.018)	Loss 0.4511 (0.8252)	Prec 84.375% (73.305%)
Validation starts
Test: [0/79]	Time 5.153 (5.153)	Loss 0.7274 (0.7274)	Prec 78.906% (78.906%)
 * Prec 76.890% 
best acc: 76.890000
Epoch: [1][0/391]	Time 5.928 (5.928)	Data 5.802 (5.802)	Loss 0.4760 (0.4760)	Prec 84.375% (84.375%)
Epoch: [1][100/391]	Time 0.034 (0.092)	Data 0.001 (0.058)	Loss 0.5310 (0.4789)	Prec 84.375% (83.826%)
Epoch: [1][200/391]	Time 0.033 (0.063)	Data 0.001 (0.029)	Loss 0.5938 (0.4722)	Prec 80.469% (84.111%)
Epoch: [1][300/391]	Time 0.034 (0.053)	Data 0.000 (0.020)	Loss 0.4501 (0.4627)	Prec 85.156% (84.585%)
Validation starts
Test: [0/79]	Time 5.517 (5.517)	Loss 0.

## Try $\gamma=10^{-12}$

In [20]:
model5 = VGG16()
model5.load_state_dict(model.state_dict())
model_name5 = "VGG16_loss3_gamma_e-6"

In [21]:
lr = 4e-2
weight_decay = 1e-4
epochs = 20
best_prec = 0
gamma = 1e-12
#model = nn.DataParallel(model).cuda()
model5.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model5.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name5)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train3(trainloader, model5, criterion, optimizer, epoch, gamma)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate3(testloader, model5, criterion, gamma)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model5.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 5.907 (5.907)	Data 5.777 (5.777)	Loss 0.0474 (0.0474)	Prec 98.438% (98.438%)
Epoch: [0][100/391]	Time 0.034 (0.095)	Data 0.001 (0.058)	Loss 0.6315 (0.9801)	Prec 79.688% (69.268%)
Epoch: [0][200/391]	Time 0.032 (0.064)	Data 0.001 (0.030)	Loss 0.5841 (0.8327)	Prec 81.250% (73.352%)
Epoch: [0][300/391]	Time 0.034 (0.054)	Data 0.001 (0.020)	Loss 0.6755 (0.7609)	Prec 75.781% (75.345%)
Validation starts
Test: [0/79]	Time 5.825 (5.825)	Loss 0.5329 (0.5329)	Prec 80.469% (80.469%)
 * Prec 80.180% 
best acc: 80.180000
Epoch: [1][0/391]	Time 5.660 (5.660)	Data 5.530 (5.530)	Loss 0.5378 (0.5378)	Prec 77.344% (77.344%)
Epoch: [1][100/391]	Time 0.033 (0.091)	Data 0.001 (0.055)	Loss 0.3854 (0.4563)	Prec 87.500% (84.746%)
Epoch: [1][200/391]	Time 0.033 (0.062)	Data 0.001 (0.028)	Loss 0.4021 (0.4533)	Prec 85.156% (84.581%)
Epoch: [1][300/391]	Time 0.033 (0.053)	Data 0.001 (0.019)	Loss 0.4276 (0.4403)	Prec 85.938% (85.078%)
Validation starts
Test: [0/79]	Time 5.585 (5.585)	Loss 0.

## Get the Absolute Summed Weights

In [22]:
og_sum = model.features[0].weight.abs().sum()
sum2 = model2.features[0].weight.abs().sum()
sum3 = model3.features[0].weight.abs().sum()
sum4 = model4.features[0].weight.abs().sum()
sum5 = model5.features[0].weight.abs().sum()

print("Weights:")
print(f"Original Model: {og_sum.item():.4f}")
print(f"Model 2:        {sum2.item():.4f}")
print(f"Model 3:        {sum3.item():.4f}")
print(f"Model 4:        {sum4.item():.4f}")
print(f"Model 5:        {sum5.item():.4f}")

Weights:
Original Model: 202.0988
Model 2:        75.1608
Model 3:        13.1602
Model 4:        331.7732
Model 5:        337.2754


# Part 4

| Case | Cost Function | Test Accuracy (%) | Conv1 Weight Abs. Sum |
| ---- | -------- | ----| -----------|
| Initial Version | loss1 | 90.290 |   202.0988 |
| New Loss Term ($\gamma=1$) | loss1 + loss2 | 77.050  | 75.1608 | 
| Fine-tuned (optimal $\gamma$) | loss1 + $\gamma$ loss2 | 82.560  |  13.1602 |

**NOTE: The optimal $\gamma=10^{-1}=0.1$**

### Discussion
The addition of the loss2 term seems to decrease the conv1 weight absolute sum (as expected). Interestingly, the $\gamma = 1$ case did not have a lower absolute summed weight comparative to our "optimized" $\gamma=0.1$. Noticeably, as we decrease $\gamma$, we also see a increase in accuracy which is expected, as we get a closer loss calculation to our original VGG16 model.    
   
We can see that given our "optimal" model sacrifices about 7% of accuracy for a 93% drop in conv1's weight absolute sum. Notably, the optimized model still manages an above 80% accuracy while still achieving substantial weight reduction compared to the smaller ($\gamma$ values tested).
